# Aula 2 — Tokenização e normalização

**Como transformar texto bruto em unidades analisáveis sem destruir informação importante**

Na Aula 1, tratamos texto como dado e aprendemos a observar documentos, metadados e problemas básicos de qualidade.

Agora damos o próximo passo: antes de representar texto numericamente, precisamos decidir **como separar o texto em unidades** e **quais transformações aplicar antes da análise**.

Essas decisões parecem simples, mas podem alterar de forma importante o que um modelo ou análise conseguirá enxergar.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar o que é tokenização;
- explicar o que é normalização textual;
- comparar texto original e texto normalizado;
- perceber como pontuação, caixa e acentuação alteram os tokens;
- construir uma tokenização simples com Python;
- reconhecer que pré-processamento envolve escolhas e trade-offs, não apenas limpeza automática.


## 📘 Glossário da aula

Conceitos centrais desta aula: **texto bruto · token · tokenização · normalização · expressão regular**.

Use o Glossário Vivo quando quiser revisar uma definição, conferir o termo técnico em inglês ou retomar a relação entre conceitos.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)

> O notebook explica o necessário para seguir a aula; o glossário serve para consolidar e aprofundar o vocabulário técnico.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.

```text
Notebook oficial = referência do curso
Cópia do aluno    = ambiente pessoal de aprendizagem
```


## 2. Por que não entregamos texto bruto diretamente a qualquer algoritmo?

Considere estas frases:

- `O atendimento foi excelente!`
- `o atendimento foi excelente`
- `O ATENDIMENTO foi excelente!!!`

Para uma pessoa, elas expressam praticamente a mesma ideia. Para uma representação baseada em formas superficiais, porém, `O`, `o` e `ATENDIMENTO` podem aparecer como elementos diferentes.

**Por que isso importa?** Porque a forma como transformamos o texto determina quais padrões ficam visíveis e quais informações podem ser perdidas.


## 3. Dois conceitos fundamentais

### Tokenização

É o processo de dividir um texto em unidades menores chamadas **tokens**.

Dependendo da técnica, um token pode ser uma palavra, parte de uma palavra, caractere ou outro tipo de unidade.

Nesta aula começaremos com uma tokenização simples baseada em palavras.

### Normalização

É o conjunto de transformações utilizadas para reduzir variações superficiais do texto.

Exemplos:

- converter para minúsculas;
- remover espaços extras;
- tratar pontuação;
- eventualmente tratar acentos ou outras variações.

> Importante: normalizar não significa remover tudo que parece diferente. Algumas diferenças carregam informação útil.


## 4. Nosso pequeno experimento

Vamos usar algumas frases curtas e observar como pequenas transformações alteram a representação do texto.

Execute a próxima célula para criar o corpus de exemplo.


In [ ]:
texts = [
    "O atendimento foi excelente!",
    "o atendimento foi excelente",
    "O ATENDIMENTO foi excelente!!!",
    "Não gostei do atendimento.",
    "Não gostei do atendimento...",
]

for i, text in enumerate(texts, start=1):
    print(i, repr(text))


### O que observar

As frases são semanticamente próximas em alguns casos, mas apresentam diferenças de:

- uso de maiúsculas e minúsculas;
- quantidade de pontuação;
- presença de negação;
- forma visual da sentença.

**Checkpoint 1:** identifique quais diferenças parecem apenas gráficas e quais podem alterar o significado.


## 5. Uma primeira normalização

Vamos aplicar apenas duas transformações seguras para este experimento:

1. converter o texto para minúsculas;
2. remover espaços extras nas extremidades.

Execute a célula abaixo e compare com o texto original.


In [ ]:
normalized_texts = [text.lower().strip() for text in texts]

for original, normalized in zip(texts, normalized_texts):
    print("ORIGINAL:   ", original)
    print("NORMALIZADO:", normalized)
    print("-")


### O que interpretar

A normalização reduziu parte da variação visual, mas **não removeu a pontuação nem alterou palavras**.

Isso é intencional. O objetivo aqui é mostrar que pré-processamento deve ser incremental e justificável.

**Checkpoint 2:** confirme que `O`, `o` e `ATENDIMENTO` deixaram de produzir variações de caixa, mas a pontuação continua visível.


## 6. Tokenização simples com Python

Para começar, vamos usar o método `.split()` do próprio Python.

Ele separa o texto por espaços. É uma estratégia simples e útil para aprender o conceito, embora não seja suficiente para todos os casos reais.


In [ ]:
tokens_by_text = [text.split() for text in normalized_texts]

for text, tokens in zip(normalized_texts, tokens_by_text):
    print("TEXTO :", text)
    print("TOKENS:", tokens)
    print("-")


### O que observar

Veja que a tokenização com `.split()` preserva a pontuação grudada às palavras:

- `excelente!`
- `excelente!!!`
- `atendimento.`

Isso significa que uma tokenização ingênua pode considerar essas formas como tokens diferentes.

**Por que isso importa?** Porque contagens, vocabulários e modelos baseados em frequência podem tratar essas variações como itens distintos.


## 7. Tokenização com expressão regular

Agora vamos usar a biblioteca padrão `re` para criar uma tokenização um pouco mais controlada.

A expressão `\b\w+\b` procura sequências de caracteres alfanuméricos delimitadas por fronteiras de palavra.

Execute a célula abaixo e compare os resultados com `.split()`.


In [ ]:
import re

regex_tokens = [re.findall(r"\b\w+\b", text) for text in normalized_texts]

for text, tokens in zip(normalized_texts, regex_tokens):
    print("TEXTO :", text)
    print("TOKENS:", tokens)
    print("-")


### O que interpretar

Agora `excelente!` e `excelente!!!` resultam no token `excelente`.

Mas isso não significa que a expressão regular seja sempre superior. Em alguns problemas, pontuação, hashtags, emojis, hífens ou símbolos podem carregar significado.

A pergunta correta não é:

> Qual é a melhor tokenização em geral?

Mas sim:

> Qual tokenização preserva a informação relevante para o meu problema?


## 8. Exercício guiado

Agora você vai construir uma pequena função de normalização e tokenização.

Sua função deve:

1. receber uma string;
2. converter para minúsculas;
3. remover espaços nas extremidades;
4. retornar tokens sem pontuação simples usando expressão regular.

Teste a função com:

```text
"  O atendimento foi EXCELENTE!!!  "
```


In [ ]:
# Escreva sua solução aqui.

def normalize_and_tokenize(text):
    # seu código
    pass

example = "  O atendimento foi EXCELENTE!!!  "
print(normalize_and_tokenize(example))


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q2.hint()` e `q2.solution()`.

A dica indica a biblioteca e os métodos relevantes sem entregar imediatamente o código final.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q2 = TILExercise(
    hint_text=(
        "Use **Python + biblioteca padrão `re`**. "
        "Você pode combinar `text.lower()`, `text.strip()` e `re.findall(...)`. "
        "A expressão regular `r'\\b\\w+\\b'` pode ser usada para extrair sequências de palavras sem a pontuação simples."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "import re\n\n"
        "def normalize_and_tokenize(text):\n"
        "    normalized = text.lower().strip()\n"
        "    return re.findall(r'\\b\\w+\\b', normalized)\n\n"
        "example = '  O atendimento foi EXCELENTE!!!  '\n"
        "print(normalize_and_tokenize(example))\n"
        "```\n\n"
        "Resultado esperado: `['o', 'atendimento', 'foi', 'excelente']`."
    ),
)

print("Exercício preparado. Tente resolver antes de usar q2.hint() ou q2.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q2.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q2.solution()


## 9. Reprodutibilidade

Esta aula não depende de datasets externos nem de bibliotecas adicionais.

- linguagem: Python;
- bibliotecas: padrão do Python (`re`);
- acelerador: CPU;
- internet: desabilitada;
- corpus: criado dentro do próprio notebook.

Isso torna o experimento simples de reproduzir e ajuda a isolar os efeitos das transformações estudadas.


## 10. Resumo

Nesta aula, você aprendeu que:

- tokenização divide texto em unidades menores;
- normalização reduz variações superficiais;
- `.split()` é útil para aprender, mas tem limitações;
- expressões regulares permitem controlar melhor a extração de tokens;
- pontuação, caixa e outras escolhas de pré-processamento podem alterar o vocabulário observado;
- pré-processamento deve preservar a informação relevante para o problema.

### Ideia principal

```text
Pré-processar texto não é apagar diferenças.
É decidir conscientemente quais diferenças importam.
```

**Fim da Aula 2.**
